# Test notebook for DPD funcionality

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import LJChain

molecules = LJChain(
    num_mols=500,
    lengths=100,
    bead_sequence=["_A"],
    bead_mass={"_A": 1.0},
    bond_lengths={"_A-_A": 1.0},
)

## New Random Walk System Class

In [3]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 1.0 * u.Unit("nm")  
ref_mass = 1.0 * u.Unit("g/mol")
ref_energy = 1.0 * u.Unit("kcal / mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.1 * u.Unit("nm**-3"),
    bond_length=1.0,
    buffer=0.5,
    base_units=ref_values_dict,
)

(50000, 3)


## FF from GMSO XML file

In [4]:
from flowermd.library.forcefields import Bead_Spring_DPD

In [5]:
system.apply_forcefield(force_field=Bead_Spring_DPD(), r_cut=1.01, kT=1.0, speedup_by_molgraph=True, speedup_by_moltag=False)

## Forcefield class

In [6]:
from flowermd.library import PhantomWalk

sim = PhantomWalk(
    initial_state=system.hoomd_snapshot,
    forcefield=system.hoomd_forcefield,
    gsd_write_freq=100,
    log_write_freq=50,
    n_steps_dpd=500,
    n_steps_fire=100,
)

Initializing simulation state from a gsd.hoomd.Frame.
Step 75 of 500; TPS: 30.66; ETA: 0.2 minutes
Step 150 of 500; TPS: 31.94; ETA: 0.2 minutes
Step 225 of 500; TPS: 34.27; ETA: 0.1 minutes
Step 300 of 500; TPS: 37.5; ETA: 0.1 minutes
Step 375 of 500; TPS: 40.3; ETA: 0.1 minutes
Step 450 of 500; TPS: 42.33; ETA: 0.0 minutes
Step 25 of 100; TPS: 105.05; ETA: 0.0 minutes


In [7]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [8]:
# system.to_gsd("random_walk_test.gsd")